# 12 - Overlay: ticker mention *first derivative* vs price (AUTO, weekly)

Auto-picks the most-mentioned tickers; plots the change in attention (the
first derivative of the mention total) vs price. `FREQ='W'` gives a smoother
weekly derivative.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window: the PIPELINE_* env vars (set by update_data.py, including its
# --start/--end overrides) win; otherwise fall back to the constants at the
# top of update_data.py. Same toggle as live vs backtest either way.
import update_data
START_DATE = os.environ.get('PIPELINE_START_DATE') or update_data.START_DATE
END_DATE = os.environ.get('PIPELINE_END_DATE')
if END_DATE is None:
    END_DATE = update_data.END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    # daily close, then made CONTINUOUS (forward-fill weekends/holidays) so the
    # line is smooth with no gaps. Clip to the window.
    one = prices[prices['symbol'] == symbol].sort_values('date')
    s = one.set_index('date')['px_last']
    if not s.empty:
        s = s.asfreq('D').ffill()
    return clip_series(s)


In [ ]:
HOW_MANY = 6
FREQ = 'W'        # 'W' weekly, 'D' daily, 'M' monthly
WINDOW = 7        # rolling window (only used when FREQ='D')

In [ ]:
counts = pd.read_parquet(os.path.join(P, 'daily_ticker_counts.parquet'))
counts['date'] = pd.to_datetime(counts['date']); counts = clip_dates(counts, 'date')
prices = load_prices()
priced = set(prices['symbol'].unique())
ranked = counts.groupby('ticker')['mention_count'].sum().sort_values(ascending=False)
unpriced = [t for t in ranked.head(HOW_MANY).index if t not in priced]
if unpriced:
    print('top names MISSING from prices.parquet (re-run pull_bloomberg_prices.py):', unpriced)
tickers = [t for t in ranked.index if t in priced][:HOW_MANY]
print('auto tickers (top mentioned WITH price data):', tickers)
for ticker in tickers:
    m = counts[counts['ticker'] == ticker].sort_values('date').set_index('date')['mention_count'].asfreq('D').fillna(0)
    px = price_series(prices, ticker)
    if px.empty:
        print('skip', ticker, '- no price rows'); continue
    if FREQ == 'D':
        deriv = m.rolling(WINDOW).sum().diff()
    else:
        deriv = m.resample(FREQ).sum().diff(); px = px.resample(FREQ).last()
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.axhline(0, color='black', linewidth=0.6)
    ax1.plot(deriv.index, deriv.values, color='tab:green', linewidth=1.7, label='1st derivative')
    ax1.set_ylabel(f'change in mentions per {FREQ}', color='tab:green'); ax1.tick_params(axis='y', labelcolor='tab:green')
    ax2 = ax1.twinx(); ax2.plot(px.index, px.values, color='tab:red', linewidth=1.8, label='price')
    ax2.set_ylabel('price (USD)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
    ax1.set_title(f'{ticker}: attention first derivative vs price ({FREQ})'); ax1.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()